# Polars Exercises 08 — Cleaning Data

Raw exports are messy: duplicate rows, text in different cases, stray spaces, numbers stored as strings (`"$3.50"`, `"10%"`, `"N/A"`), and dates in three different formats. Before *any* analysis is trustworthy, the data must be cleaned. This is one of the most important real-world skills — and a big part of what our Foundry transforms actually do.


**Data file:** `data/orders_messy.csv`

---

### 1. Import Polars and read `data/orders_messy.csv` into `messy`. Look at the first rows — notice the dirty values.

In [1]:
import polars as pl

messy = pl.read_csv("data/orders_messy.csv")
messy.head()

order_id,customer_id,order_date,region,category,quantity,unit_price,discount,status
str,str,str,str,str,str,str,str,str
"""ORD-000181""","""CUST-0060""","""2023.11.18""",""" Europe """,""" posters""","""N/A""","""$6.86""","""""","""Completed"""
"""ORD-000194""","""CUST-0004""","""2023-07-27""","""NORTH AMERICA""","""PENS""","""29""","""$1.22""","""""","""Completed"""
"""ORD-000150""","""CUST-0001""","""2023-08-22""","""asia pacific""",""" pens""","""34""","""$1.27""","""25%""","""Completed"""
"""ORD-000130""","""CUST-0184""","""2023-02-13""","""europe""","""SHIRTS""","""22""","""$18.36""","""""","""COMPLETED"""
"""ORD-000274""","""CUST-0152""","""01/01/2023""",""" North America ""","""NOTEBOOKS""","""34""","""$3.87""","""""","""Completed"""


### 2. Check the schema. Which columns came in as **strings** even though they should be numbers or dates?

In [2]:
messy.schema

Schema([('order_id', String),
        ('customer_id', String),
        ('order_date', String),
        ('region', String),
        ('category', String),
        ('quantity', String),
        ('unit_price', String),
        ('discount', String),
        ('status', String)])

### 3. How many **fully-duplicate rows** are there? (hint: `height` minus `unique().height`)

In [3]:
messy.height - messy.unique().height

25

### 4. Remove the exact duplicate rows with `unique()`. How many rows remain?

In [4]:
dedup = messy.unique()
dedup.height

300

### 5. Clean the `region` column: **strip** surrounding spaces and convert to **Title Case** (so `"  EUROPE "` and `"europe"` both become `"Europe"`).

In [5]:
messy.with_columns(
    region=pl.col("region").str.strip_chars().str.to_titlecase()
).get_column("region").unique().sort()

region
str
"""Asia Pacific"""
"""Europe"""
"""Latin America"""
"""Middle East & Africa"""
"""North America"""


### 6. Clean the `category` column to a consistent lower-case, with no surrounding spaces.

In [6]:
messy.with_columns(
    category=pl.col("category").str.strip_chars().str.to_lowercase()
).get_column("category").unique().sort()

category
str
"""books"""
"""mugs"""
"""notebooks"""
"""pens"""
"""posters"""
"""shirts"""
"""stickers"""


### 7. Fix `quantity`: it is a string with some `"N/A"` values. Cast it to an integer with `strict=False` so bad values become **null**.

In [7]:
messy.with_columns(
    quantity=pl.col("quantity").cast(pl.Int64, strict=False)
).get_column("quantity").head(10)

quantity
i64
null
29
34
22
34
13
24
30
8


### 8. **How many** `quantity` values were invalid (became null after the cast)?

In [8]:
messy.select(
    pl.col("quantity").cast(pl.Int64, strict=False).is_null().sum()
)

quantity
u32
15


### 9. Fix `unit_price`: it looks like `"$3.50"`. Strip the leading `$` and cast to a float.

In [9]:
messy.with_columns(
    unit_price=pl.col("unit_price").str.strip_chars_start("$").cast(pl.Float64)
).get_column("unit_price").head()

unit_price
f64
6.86
1.22
1.27
18.36
3.87
1.16
9.7
1.12
8.54


### 10. Fix `discount`: values look like `"10%"` (and some are empty). Turn them into a fraction (`0.10`); empty values should become null. (strip `%`, cast to float with `strict=False`, divide by 100)

In [10]:
messy.with_columns(
    discount=pl.col("discount").str.strip_chars("%")
    .cast(pl.Float64, strict=False) / 100
).get_column("discount").head(10)

discount
f64
null
null
0.25
null
null
0.16
null
null
null


### 11. Fix `order_date`: it arrives in **three** formats — `2023-01-05`, `05/01/2023`, `2023.01.05`. Parse all of them into one real Date column using `pl.coalesce` over three `to_date` attempts.

In [11]:
messy.with_columns(
    order_date=pl.coalesce(
        pl.col("order_date").str.to_date("%Y-%m-%d", strict=False),
        pl.col("order_date").str.to_date("%d/%m/%Y", strict=False),
        pl.col("order_date").str.to_date("%Y.%m.%d", strict=False),
    )
).get_column("order_date").head()

order_date
date
2023-11-18
2023-07-27
2023-08-22
2023-02-13
2023-01-01
2023-09-16
2023-09-26
2023-08-08
2023-11-09


### 12. Standardize `status` to lower-case, then show the `value_counts` to confirm there are only the expected categories.

In [12]:
messy.with_columns(
    status=pl.col("status").str.to_lowercase()
).get_column("status").value_counts(sort=True)

status,count
str,u32
"""completed""",268
"""pending""",29
"""cancelled""",15
"""returned""",13


### 13. 🎯 **Mission — the full cleaning pipeline.** In one `with_columns` chain, produce a clean table: dedupe, fix `region`, `category`, `quantity`, `unit_price`, `discount`, `order_date`, and `status` all together. Save it as `clean` and show its schema.

In [13]:
clean = messy.unique().with_columns(
    region=pl.col("region").str.strip_chars().str.to_titlecase(),
    category=pl.col("category").str.strip_chars().str.to_lowercase(),
    quantity=pl.col("quantity").cast(pl.Int64, strict=False),
    unit_price=pl.col("unit_price").str.strip_chars_start("$").cast(pl.Float64),
    discount=pl.col("discount").str.strip_chars("%").cast(pl.Float64, strict=False) / 100,
    order_date=pl.coalesce(
        pl.col("order_date").str.to_date("%Y-%m-%d", strict=False),
        pl.col("order_date").str.to_date("%d/%m/%Y", strict=False),
        pl.col("order_date").str.to_date("%Y.%m.%d", strict=False),
    ),
    status=pl.col("status").str.to_lowercase(),
)
clean.schema

Schema([('order_id', String),
        ('customer_id', String),
        ('order_date', Date),
        ('region', String),
        ('category', String),
        ('quantity', Int64),
        ('unit_price', Float64),
        ('discount', Float64),
        ('status', String)])

### 14. From the clean table, **drop the rows where `quantity` is null** (the invalid `"N/A"` rows). How many rows remain?

In [14]:
clean.drop_nulls(subset="quantity").height

285

### 15. In the clean table, fill the remaining null `discount` values with `0`.

In [15]:
clean.with_columns(
    discount=pl.col("discount").fill_null(0)
).get_column("discount").head(10)

discount
f64
0.0
0.0
0.08
0.0
0.0
0.0
0.0
0.0
0.0


### 16. Prove the data is now usable: add `revenue` (`quantity * unit_price * (1 - discount filled with 0)`) and show total revenue **per region**.

In [16]:
clean.drop_nulls(subset="quantity").with_columns(
    revenue=pl.col("quantity") * pl.col("unit_price")
    * (1 - pl.col("discount").fill_null(0))
).group_by("region").agg(pl.col("revenue").sum().round(2))\
   .sort("revenue", descending=True)

region,revenue
str,f64
"""North America""",16464.17
"""Europe""",10680.58
"""Asia Pacific""",9557.37
"""Latin America""",5864.21
"""Middle East & Africa""",1125.46
